In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os, sys
project_path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_path)
print(project_path)



In [0]:
import importlib
import utils.transformation
importlib.reload(utils.transformation)

from utils.transformation import reusable


### Dim User

## AUTOLOADER 

In [0]:
checkpoint_base = "abfss://silver@storagespotify2201.dfs.core.windows.net/DimUser"

# Step 1 — Read stream
df_user = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@storagespotify2201.dfs.core.windows.net/DimUser")



In [0]:
df_user.printSchema()

In [0]:
# Step 2 — Apply transformations
df_user = df_user.withColumn("user_name", upper(col("user_name")))

In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])



In [0]:
#display(df_user, checkpointLocation = f"{checkpoint_base}/checkpoint")

In [0]:

#dbutils.fs.rm("abfss://silver@storagespotify2201.dfs.core.windows.net/DimUser/checkpoint", recurse=True)

In [0]:
df_user.writeStream.format("delta")\
       .outputMode("append")\
       .option("checkpointLocation","abfss://silver@storagespotify2201.dfs.core.windows.net/DimUser/checkpoint")\
       .trigger(once=True)\
       .option("path","abfss://silver@storagespotify2201.dfs.core.windows.net/DimUser/data")\
       .toTable("spotify_cata.silver.dim_user")


## DimArtist

In [0]:

#dbutils.fs.rm(
    "abfss://silver@storagespotify2201.dfs.core.windows.net/DimArtist/checkpoint", 
    recurse=True
#)
#print("Checkpoint cleared ✅")

In [0]:

checkpoint_base_artist = "abfss://silver@storagespotify2201.dfs.core.windows.net/DimArtist"

# readStream
df_artist = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", f"{checkpoint_base_artist}/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@storagespotify2201.dfs.core.windows.net/DimArtist")

# Clean transforms only — no dropDuplicates
df_artist_obj = reusable()
df_artist = df_artist_obj.dropColumns(df_artist, ['_rescued_data'])

# Write stream
df_artist.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base_artist}/checkpoint") \
    .trigger(availableNow=True) \
    .option("path",f"{checkpoint_base_artist}/data")\
    .toTable("spotify_cata.silver.dim_artist")

## Dim Track


In [0]:
checkpoint_base_track = "abfss://silver@storagespotify2201.dfs.core.windows.net/DimTrack"

# Step 1 — Read stream
df_track = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", f"{checkpoint_base_track}/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@storagespotify2201.dfs.core.windows.net/DimTrack")

In [0]:
df_track.printSchema()

In [0]:
df_track = df_track.withColumn("durationflag", when(col('duration_sec')<150, "low")\
                                        .when(col("duration_sec")<300, "medium")\
                                        .otherwise("high"))

df_track = df_track.withColumn("track_name", regexp_replace(col('track_name'),'-',''))

df_track = reusable().dropColumns(df_track,['_rescued_data'])




In [0]:
df_track.printSchema()

In [0]:

dbutils.fs.rm(
    "abfss://silver@storagespotify2201.dfs.core.windows.net/DimTrack/checkpoint", 
    recurse=True
)
print("Checkpoint cleared ✅")

In [0]:
# Write stream
df_track.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base_track}/checkpoint") \
    .trigger(availableNow=True) \
    .option("path",f"{checkpoint_base_track}/data")\
    .toTable("spotify_cata.silver.dim_track")

## DimDate

In [0]:
checkpoint_base_date = "abfss://silver@storagespotify2201.dfs.core.windows.net/DimDate"

# Step 1 — Read stream
df_date = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", f"{checkpoint_base_date}/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@storagespotify2201.dfs.core.windows.net/DimDate")

In [0]:
df_date.printSchema()

In [0]:
df_date = reusable().dropColumns(df_date,['_rescued_data'])



In [0]:
df_date.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base_date}/checkpoint") \
    .trigger(availableNow=True) \
    .option("path",f"{checkpoint_base_date}/data")\
    .toTable("spotify_cata.silver.dim_date")


## FactStream

In [0]:
checkpoint_base_stream = "abfss://silver@storagespotify2201.dfs.core.windows.net/FactStream"

# Step 1 — Read stream
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", f"{checkpoint_base_stream}/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@storagespotify2201.dfs.core.windows.net/FactStream")

In [0]:
df_fact = df_stream

In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])


In [0]:
df_fact.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_base_stream}/checkpoint") \
    .trigger(availableNow=True) \
    .option("path",f"{checkpoint_base_stream}/data")\
    .toTable("spotify_cata.silver.FactStream")

In [0]:
%sql
SELECT * FROM spotify_cata.gold.dim_track
WHERE track_id IN (46,5)
